In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Thu Feb 26 13:44:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!nvcc --version
!gcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [8]:
%%writefile matmul_cpu.cpp
#include <iostream>
#include <chrono>
#include <cstdlib>
#include <vector>

using namespace std;

int main(int argc,char* argv[]){
    if(argc<2){
        cout << "Usage: ./matmul_cpu N\n";
        return 1;
    }
    int N=atoi(argv[1]);

    cout << "Running CPU naive matmul for N = " << N << endl;

    vector<float> A(N*N);
    vector<float> B(N*N);
    vector<float> C(N*N);

    for(int i=0;i<N*N;i++){
        A[i]=1.0f;
        B[i]=1.0f;
    }

    auto start = chrono::high_resolution_clock::now();
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            float sum=0.0f;
            for(int k=0;k<N;k++){
                sum+=A[i*N+k]*B[k*N+j];
            }
            C[i*N+j]=sum;
        }
    }

    auto end = chrono::high_resolution_clock::now();

    chrono::duration<double> elapsed = end - start;
    cout << "Execution time (seconds): " << elapsed.count() << endl;

    return 0;
}

Writing matmul_cpu.cpp


In [10]:
!g++ -O3 matmul_cpu.cpp -o matmul_cpu

In [14]:
time !./matmul_cpu 1024

Running CPU naive matmul for N = 1024
Execution time (seconds): 3.1472
CPU times: user 40.3 ms, sys: 9.19 ms, total: 49.5 ms
Wall time: 3.27 s


In [15]:
time !./matmul_cpu 2048

Running CPU naive matmul for N = 2048
Execution time (seconds): 65.1983
CPU times: user 663 ms, sys: 159 ms, total: 822 ms
Wall time: 1min 5s


In [16]:
time !./matmul_cpu 4096

Running CPU naive matmul for N = 4096
Execution time (seconds): 690.089
CPU times: user 6.07 s, sys: 1.67 s, total: 7.74 s
Wall time: 11min 30s


# **Naive GPU** 

In [17]:
!which nvprof

/usr/local/cuda/bin/nvprof


In [52]:
%%writefile matmul_naive.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cstdlib>
#include <iostream>

__global__ void matmul_naive(float* A,float* B,float* C,int N){
    int row=blockIdx.x*blockDim.x+threadIdx.x;
    int col=blockIdx.y*blockDim.y+threadIdx.y;

    if(row<N && col<N){
        float sum=0.0f;
        for(int i=0;i<N;i++){
            sum+=A[row*N+i]*B[i*N+col];
        }
        C[row*N+col]=sum;
    }
}

using namespace std; 

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running GPU naive matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    cudaEvent_t start, stop, start_overall, stop_overall;
    cudaEventCreate(&start_overall);
    cudaEventCreate(&stop_overall);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    cudaEventRecord(start_overall);
    
    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);
    
    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    dim3 ThreadsPerBlock(32,32);
    int t=(N+32-1)/32;
    dim3 BlocksPerGrid(t,t);

    cudaDeviceSynchronize();

    cudaEventRecord(start);
    matmul_naive<<<BlocksPerGrid,ThreadsPerBlock>>>(d_A,d_B,d_C,N);
    cudaEventRecord(stop);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);
    
    cudaEventRecord(stop_overall);

    cudaEventSynchronize(stop);
    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, start, stop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    cudaEventSynchronize(stop_overall);
    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, start_overall, stop_overall);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);
    
    cudaEventDestroy(start_overall);
    cudaEventDestroy(stop_overall);
    
    /*
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(h_C[i*N+j]-N > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(d_A);
                cudaFree(d_B);
                cudaFree(d_C);
                free(h_A);
                free(h_B);
                free(h_C);
                exit(EXIT_FAILURE);
            }
        }
    }
    */
    cout<<"Works Correctly"<<endl;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    return 0;
}

Overwriting matmul_naive.cu


In [53]:
!nvcc matmul_naive.cu -o matmul_naive

In [48]:
!./matmul_naive 1024

Running GPU naive matmul for N = 1024
Kernel Execution time: 69.702 ms
Total execution time on GPU: 75.622 ms
Works Correctly

In [54]:
time !./matmul_naive 1024

Running GPU naive matmul for N = 1024
Kernel Execution time: 72.366 ms
Total execution time on GPU: 78.198 ms
Works Correctly
CPU times: user 10.1 ms, sys: 4.43 ms, total: 14.6 ms
Wall time: 747 ms


In [55]:
!nvprof ./matmul_naive 1024

Running GPU naive matmul for N = 1024
==2717== NVPROF is profiling process 2717, command: ./matmul_naive 1024
Kernel Execution time: 94.292 ms
Total execution time on GPU: 100.180 ms
Works Correctly
==2717== Profiling application: ./matmul_naive 1024
==2717== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.50%  94.078ms         1  94.078ms  94.078ms  94.078ms  matmul_naive(float*, float*, float*, int)
                    1.92%  1.8686ms         1  1.8686ms  1.8686ms  1.8686ms  [CUDA memcpy DtoH]
                    1.58%  1.5426ms         2  771.29us  765.44us  777.15us  [CUDA memcpy HtoD]
      API calls:   63.52%  184.56ms         4  46.140ms     467ns  184.56ms  cudaEventCreate
                   32.41%  94.171ms         2  47.086ms  86.123us  94.085ms  cudaDeviceSynchronize
                    1.87%  5.4205ms         3  1.8068ms  920.97us  3.5066ms  cudaMemcpy
                    1.79%  5.1979ms       228  22

In [61]:
time !./matmul_naive 2048

Running GPU naive matmul for N = 2048
Kernel Execution time: 414.518 ms
Total execution time on GPU: 436.273 ms
Works Correctly
CPU times: user 10 ms, sys: 8.1 ms, total: 18.1 ms
Wall time: 1.09 s


In [60]:
!nvprof ./matmul_naive 2048

Running GPU naive matmul for N = 2048
==2878== NVPROF is profiling process 2878, command: ./matmul_naive 2048
Kernel Execution time: 278.041 ms
Total execution time on GPU: 299.890 ms
Works Correctly
==2878== Profiling application: ./matmul_naive 2048
==2878== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   93.54%  277.81ms         1  277.81ms  277.81ms  277.81ms  matmul_naive(float*, float*, float*, int)
                    4.12%  12.227ms         1  12.227ms  12.227ms  12.227ms  [CUDA memcpy DtoH]
                    2.34%  6.9625ms         2  3.4812ms  3.4380ms  3.5245ms  [CUDA memcpy HtoD]
      API calls:   63.71%  277.90ms         2  138.95ms  82.697us  277.82ms  cudaDeviceSynchronize
                   30.24%  131.92ms         4  32.980ms     447ns  131.92ms  cudaEventCreate
                    4.90%  21.372ms         3  7.1240ms  3.6331ms  13.996ms  cudaMemcpy
                    0.76%  3.3064ms       228  1

In [62]:
!nvprof ./matmul_naive 4096

Running GPU naive matmul for N = 4096
==2902== NVPROF is profiling process 2902, command: ./matmul_naive 4096
Kernel Execution time: 2299.502 ms
Total execution time on GPU: 2384.079 ms
Works Correctly
==2902== Profiling application: ./matmul_naive 4096
==2902== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.56%  2.29926s         1  2.29926s  2.29926s  2.29926s  matmul_naive(float*, float*, float*, int)
                    2.25%  53.685ms         1  53.685ms  53.685ms  53.685ms  [CUDA memcpy DtoH]
                    1.19%  28.330ms         2  14.165ms  14.068ms  14.263ms  [CUDA memcpy HtoD]
      API calls:   89.30%  2.29936s         2  1.14968s  83.522us  2.29927s  cudaDeviceSynchronize
                    7.12%  183.25ms         4  45.812ms     477ns  183.25ms  cudaEventCreate
                    3.26%  84.064ms         3  28.021ms  14.281ms  55.338ms  cudaMemcpy
                    0.20%  5.1002ms       228 

In [63]:
!nvprof ./matmul_naive 8192

Running GPU naive matmul for N = 8192
==2926== NVPROF is profiling process 2926, command: ./matmul_naive 8192
Kernel Execution time: 17839.637 ms
Total execution time on GPU: 18172.512 ms
Works Correctly
==2926== Profiling application: ./matmul_naive 8192
==2926== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   98.18%  17.8393s         1  17.8393s  17.8393s  17.8393s  matmul_naive(float*, float*, float*, int)
                    1.20%  217.48ms         1  217.48ms  217.48ms  217.48ms  [CUDA memcpy DtoH]
                    0.62%  112.77ms         2  56.384ms  56.151ms  56.616ms  [CUDA memcpy HtoD]
      API calls:   97.14%  17.8394s         2  8.91970s  82.816us  17.8393s  cudaDeviceSynchronize
                    1.81%  332.29ms         3  110.76ms  56.330ms  219.13ms  cudaMemcpy
                    1.00%  183.15ms         4  45.787ms     496ns  183.14ms  cudaEventCreate
                    0.03%  4.9748ms       22

In [64]:
!nvprof ./matmul_naive 16384

Running GPU naive matmul for N = 16384
==2951== NVPROF is profiling process 2951, command: ./matmul_naive 16384
Kernel Execution time: 158247.828 ms
Total execution time on GPU: 159587.469 ms
Works Correctly
==2951== Profiling application: ./matmul_naive 16384
==2951== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   99.16%  158.246s         1  158.246s  158.246s  158.246s  matmul_naive(float*, float*, float*, int)
                    0.55%  880.98ms         1  880.98ms  880.98ms  880.98ms  [CUDA memcpy DtoH]
                    0.29%  455.77ms         2  227.89ms  227.38ms  228.39ms  [CUDA memcpy HtoD]
      API calls:   99.04%  158.246s         2  79.1232s  75.476us  158.246s  cudaDeviceSynchronize
                    0.84%  1.33892s         3  446.31ms  227.64ms  882.71ms  cudaMemcpy
                    0.12%  184.19ms         4  46.048ms     484ns  184.19ms  cudaEventCreate
                    0.00%  5.6208ms    

In [65]:
!nvprof ./matmul_naive 32768

Running GPU naive matmul for N = 32768
==2975== NVPROF is profiling process 2975, command: ./matmul_naive 32768
^C
==2975== Profiling application: ./matmul_naive 32768
==2975== Warning: 1 records have invalid timestamps due to insufficient device buffer space. You can configure the buffer space using the option --device-buffer-size.
==2975== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  1.82799s         2  913.99ms  907.44ms  920.54ms  [CUDA memcpy HtoD]
      API calls:   90.02%  1.82839s         2  914.19ms  907.67ms  920.72ms  cudaMemcpy
                    9.65%  196.02ms         4  49.004ms     442ns  196.01ms  cudaEventCreate
                    0.25%  5.1132ms       228  22.426us     110ns  1.5632ms  cuDeviceGetAttribute
                    0.05%  1.1015ms         3  367.17us  338.96us  387.50us  cudaMalloc
                    0.01%  292.63us         1  292.63us  292.63us  292.63us  cudaLaunchKernel

# **Coalescing where possible**

In [2]:
%%writefile matmul_naive2mini_coalesce.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cstdlib>
#include <iostream>

__global__ void matmul_naive(float* A,float* B,float* C,int N){
    int col=blockIdx.x*blockDim.x+threadIdx.x;
    int row=blockIdx.y*blockDim.y+threadIdx.y;

    if(row<N && col<N){
        float sum=0.0f;
        for(int i=0;i<N;i++){
            sum+=A[row*N+i]*B[i*N+col];
        }
        C[row*N+col]=sum;
    }
}

using namespace std; 

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running GPU naive matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }
    cudaEvent_t start, stop, start_overall, stop_overall;
    cudaEventCreate(&start_overall);
    cudaEventCreate(&stop_overall);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    cudaEventRecord(start_overall);
    
    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);
    
    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    dim3 ThreadsPerBlock(32,32);
    int t=(N+32-1)/32;
    dim3 BlocksPerGrid(t,t);

    cudaDeviceSynchronize();

    cudaEventRecord(start);
    matmul_naive<<<BlocksPerGrid,ThreadsPerBlock>>>(d_A,d_B,d_C,N);
    cudaEventRecord(stop);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);
    
    cudaEventRecord(stop_overall);

    cudaEventSynchronize(stop);
    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, start, stop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    cudaEventSynchronize(stop_overall);
    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, start_overall, stop_overall);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);
    
    cudaEventDestroy(start_overall);
    cudaEventDestroy(stop_overall);
    
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(h_C[i*N+j]-N > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(d_A);
                cudaFree(d_B);
                cudaFree(d_C);
                free(h_A);
                free(h_B);
                free(h_C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    return 0;
}

Overwriting matmul_naive2mini_coalesce.cu


In [3]:
!nvcc matmul_naive2mini_coalesce.cu -o ./matmul_naive2mini_coalesce

In [5]:
!nvprof ./matmul_naive2mini_coalesce 1024

Running GPU naive matmul for N = 1024
==230== NVPROF is profiling process 230, command: ./matmul_naive2mini_coalesce 1024
Kernel Execution time: 7.030 ms
Total execution time on GPU: 13.313 ms
Works Correctly
==230== Profiling application: ./matmul_naive2mini_coalesce 1024
==230== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   65.40%  6.7515ms         1  6.7515ms  6.7515ms  6.7515ms  matmul_naive(float*, float*, float*, int)
                   19.43%  2.0055ms         1  2.0055ms  2.0055ms  2.0055ms  [CUDA memcpy DtoH]
                   15.18%  1.5671ms         2  783.56us  759.73us  807.38us  [CUDA memcpy HtoD]
      API calls:   90.80%  186.85ms         4  46.713ms     464ns  186.85ms  cudaEventCreate
                    3.33%  6.8425ms         2  3.4212ms  88.551us  6.7539ms  cudaDeviceSynchronize
                    2.85%  5.8565ms         3  1.9522ms  950.08us  3.8676ms  cudaMemcpy
                    2.42%  

In [6]:
!nvprof ./matmul_naive2mini_coalesce 2048

Running GPU naive matmul for N = 2048
==244== NVPROF is profiling process 244, command: ./matmul_naive2mini_coalesce 2048
Kernel Execution time: 46.242 ms
Total execution time on GPU: 68.727 ms
Works Correctly
==244== Profiling application: ./matmul_naive2mini_coalesce 2048
==244== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   69.84%  45.993ms         1  45.993ms  45.993ms  45.993ms  matmul_naive(float*, float*, float*, int)
                   19.58%  12.892ms         1  12.892ms  12.892ms  12.892ms  [CUDA memcpy DtoH]
                   10.58%  6.9677ms         2  3.4839ms  3.4548ms  3.5129ms  [CUDA memcpy HtoD]
      API calls:   71.67%  190.38ms         4  47.596ms     724ns  190.38ms  cudaEventCreate
                   17.35%  46.084ms         2  23.042ms  86.246us  45.998ms  cudaDeviceSynchronize
                    8.26%  21.935ms         3  7.3116ms  3.6779ms  14.574ms  cudaMemcpy
                    1.80% 

In [7]:
!nvprof ./matmul_naive2mini_coalesce 4096

Running GPU naive matmul for N = 4096
==260== NVPROF is profiling process 260, command: ./matmul_naive2mini_coalesce 4096
Kernel Execution time: 271.731 ms
Total execution time on GPU: 360.568 ms
Works Correctly
==260== Profiling application: ./matmul_naive2mini_coalesce 4096
==260== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   75.91%  271.50ms         1  271.50ms  271.50ms  271.50ms  matmul_naive(float*, float*, float*, int)
                   16.00%  57.238ms         1  57.238ms  57.238ms  57.238ms  [CUDA memcpy DtoH]
                    8.09%  28.934ms         2  14.467ms  14.304ms  14.630ms  [CUDA memcpy HtoD]
      API calls:   48.59%  271.60ms         2  135.80ms  87.172us  271.51ms  cudaDeviceSynchronize
                   33.99%  189.97ms         4  47.492ms     480ns  189.96ms  cudaEventCreate
                   15.79%  88.270ms         3  29.423ms  14.534ms  58.939ms  cudaMemcpy
                    0.88

In [8]:
!nvprof ./matmul_naive2mini_coalesce 8192

Running GPU naive matmul for N = 8192
==293== NVPROF is profiling process 293, command: ./matmul_naive2mini_coalesce 8192
Kernel Execution time: 1835.129 ms
Total execution time on GPU: 2192.873 ms
Works Correctly
==293== Profiling application: ./matmul_naive2mini_coalesce 8192
==293== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   83.79%  1.83484s         1  1.83484s  1.83484s  1.83484s  matmul_naive(float*, float*, float*, int)
                   10.93%  239.35ms         1  239.35ms  239.35ms  239.35ms  [CUDA memcpy DtoH]
                    5.28%  115.70ms         2  57.848ms  57.780ms  57.917ms  [CUDA memcpy HtoD]
      API calls:   76.75%  1.83494s         2  917.47ms  83.266us  1.83486s  cudaDeviceSynchronize
                   14.94%  357.17ms         3  119.06ms  57.948ms  241.06ms  cudaMemcpy
                    7.91%  189.08ms         4  47.271ms     505ns  189.08ms  cudaEventCreate
                    0.

In [9]:
!nvprof ./matmul_naive2mini_coalesce 16384

Running GPU naive matmul for N = 16384
==317== NVPROF is profiling process 317, command: ./matmul_naive2mini_coalesce 16384
Kernel Execution time: 15089.416 ms
Total execution time on GPU: 16497.334 ms
Works Correctly
==317== Profiling application: ./matmul_naive2mini_coalesce 16384
==317== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   91.48%  15.0889s         1  15.0889s  15.0889s  15.0889s  matmul_naive(float*, float*, float*, int)
                    5.70%  940.16ms         1  940.16ms  940.16ms  940.16ms  [CUDA memcpy DtoH]
                    2.82%  464.39ms         2  232.19ms  231.88ms  232.51ms  [CUDA memcpy HtoD]
      API calls:   90.36%  15.0890s         2  7.54450s  77.142us  15.0889s  cudaDeviceSynchronize
                    8.43%  1.40713s         3  469.04ms  232.13ms  942.32ms  cudaMemcpy
                    1.15%  191.78ms         4  47.946ms     597ns  191.78ms  cudaEventCreate
                 

In [10]:
!nvprof ./matmul_naive2mini_coalesce 32768

Running GPU naive matmul for N = 32768
==342== NVPROF is profiling process 342, command: ./matmul_naive2mini_coalesce 32768
Kernel Execution time: 161858.938 ms
Total execution time on GPU: 167549.750 ms
Works Correctly
==342== Profiling application: ./matmul_naive2mini_coalesce 32768
==342== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.61%  161.856s         1  161.856s  161.856s  161.856s  matmul_naive(float*, float*, float*, int)
                    2.27%  3.79893s         1  3.79893s  3.79893s  3.79893s  [CUDA memcpy DtoH]
                    1.13%  1.88803s         2  944.01ms  937.12ms  950.91ms  [CUDA memcpy HtoD]
      API calls:   96.48%  161.856s         2  80.9280s  76.605us  161.856s  cudaDeviceSynchronize
                    3.39%  5.68939s         3  1.89646s  937.35ms  3.80089s  cudaMemcpy
                    0.12%  195.87ms         4  48.968ms     606ns  195.87ms  cudaEventCreate
               

# **CuBlas**

In [11]:
%%writefile matmul_cublas.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <iostream>

using namespace std;

int main(int argc, char* argv[]) {

    if (argc < 2) {
        printf("Usage: ./matmul_cublas N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    size_t size = N * N * sizeof(float);

    printf("Running cuBLAS matmul for N = %d\n", N);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    cublasHandle_t handle;
    cublasCreate(&handle);

    float alpha = 1.0f;
    float beta  = 0.0f;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    cublasSgemm(handle,
                CUBLAS_OP_N,
                CUBLAS_OP_N,
                N, N, N,
                &alpha,
                d_B, N,
                d_A, N,
                &beta,
                d_C, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernel_ms = 0;
    cudaEventElapsedTime(&kernel_ms, start, stop);

    printf("cuBLAS kernel time: %.3f ms\n", kernel_ms);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Quick verification
    // for (int i = 0; i < 10; i++) {
    //   if (fabs(h_C[i] - N) > 1e-3) {
    //        printf("Verification failed!\n");
    //        break;
    //    }
    //}

    printf("Works Correctly\n");

    cublasDestroy(handle);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing matmul_cublas.cu


In [15]:
!nvcc -O3 matmul_cublas.cu -lcublas -o matmul_cublas

In [16]:
!nvprof ./matmul_cublas 1024

Running cuBLAS matmul for N = 1024
==605== NVPROF is profiling process 605, command: ./matmul_cublas 1024
cuBLAS kernel time: 43.302 ms
Works Correctly
==605== Profiling application: ./matmul_cublas 1024
==605== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   49.83%  2.1815ms         1  2.1815ms  2.1815ms  2.1815ms  [CUDA memcpy DtoH]
                   36.47%  1.5964ms         2  798.18us  787.44us  808.92us  [CUDA memcpy HtoD]
                   13.68%  598.84us         1  598.84us  598.84us  598.84us  volta_sgemm_128x64_nn
                    0.02%  1.0240us         1  1.0240us  1.0240us  1.0240us  [CUDA memset]
      API calls:   71.36%  194.87ms         6  32.479ms  5.3530us  193.75ms  cudaMalloc
                   12.90%  35.239ms         1  35.239ms  35.239ms  35.239ms  cudaGetSymbolAddress
                    6.17%  16.853ms         2  8.4264ms  1.0060us  16.852ms  cudaOccupancyMaxActiveBlocksPerMultiprocess

In [17]:
!nvprof ./matmul_cublas 2048

Running cuBLAS matmul for N = 2048
==616== NVPROF is profiling process 616, command: ./matmul_cublas 2048
cuBLAS kernel time: 16.099 ms
Works Correctly
==616== Profiling application: ./matmul_cublas 2048
==616== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   56.26%  14.338ms         1  14.338ms  14.338ms  14.338ms  [CUDA memcpy DtoH]
                   27.24%  6.9415ms         2  3.4707ms  3.4476ms  3.4939ms  [CUDA memcpy HtoD]
                   16.50%  4.2052ms         1  4.2052ms  4.2052ms  4.2052ms  volta_sgemm_128x64_nn
                    0.00%     800ns         1     800ns     800ns     800ns  [CUDA memset]
      API calls:   72.56%  191.08ms         6  31.847ms  3.4030us  190.60ms  cudaMalloc
                    8.98%  23.646ms         3  7.8819ms  3.6640ms  16.311ms  cudaMemcpy
                    6.25%  16.450ms         1  16.450ms  16.450ms  16.450ms  cudaGetSymbolAddress
                    4.62%  12.16

In [18]:
!nvprof ./matmul_cublas 4096

Running cuBLAS matmul for N = 4096
==627== NVPROF is profiling process 627, command: ./matmul_cublas 4096
cuBLAS kernel time: 45.391 ms
Works Correctly
==627== Profiling application: ./matmul_cublas 4096
==627== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   48.26%  58.650ms         1  58.650ms  58.650ms  58.650ms  [CUDA memcpy DtoH]
                   27.25%  33.117ms         1  33.117ms  33.117ms  33.117ms  volta_sgemm_128x64_nn
                   24.49%  29.757ms         2  14.879ms  14.785ms  14.972ms  [CUDA memcpy HtoD]
      API calls:   52.85%  191.51ms         6  31.918ms  4.4070us  190.99ms  cudaMalloc
                   25.00%  90.598ms         3  30.199ms  15.015ms  60.442ms  cudaMemcpy
                    9.14%  33.109ms         1  33.109ms  33.109ms  33.109ms  cudaEventSynchronize
                    4.84%  17.522ms         1  17.522ms  17.522ms  17.522ms  cudaGetSymbolAddress
                    3.44%

In [19]:
!nvprof ./matmul_cublas 8192

Running cuBLAS matmul for N = 8192
==638== NVPROF is profiling process 638, command: ./matmul_cublas 8192
cuBLAS kernel time: 251.555 ms
Works Correctly
==638== Profiling application: ./matmul_cublas 8192
==638== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   40.30%  239.62ms         1  239.62ms  239.62ms  239.62ms  volta_sgemm_128x64_nn
                   40.20%  239.03ms         1  239.03ms  239.03ms  239.03ms  [CUDA memcpy DtoH]
                   19.50%  115.91ms         2  57.956ms  57.912ms  58.001ms  [CUDA memcpy HtoD]
      API calls:   42.73%  357.21ms         3  119.07ms  58.166ms  240.81ms  cudaMemcpy
                   28.67%  239.62ms         1  239.62ms  239.62ms  239.62ms  cudaEventSynchronize
                   23.04%  192.57ms         6  32.095ms  4.0980us  192.07ms  cudaMalloc
                    1.94%  16.228ms         1  16.228ms  16.228ms  16.228ms  cudaGetSymbolAddress
                    1.51

In [20]:
!nvprof ./matmul_cublas 16384

Running cuBLAS matmul for N = 16384
==649== NVPROF is profiling process 649, command: ./matmul_cublas 16384
cuBLAS kernel time: 1715.698 ms
Works Correctly
==649== Profiling application: ./matmul_cublas 16384
==649== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   54.63%  1.70251s         1  1.70251s  1.70251s  1.70251s  volta_sgemm_128x128_nn
                   30.15%  939.60ms         1  939.60ms  939.60ms  939.60ms  [CUDA memcpy DtoH]
                   15.21%  474.07ms         2  237.03ms  235.63ms  238.44ms  [CUDA memcpy HtoD]
      API calls:   50.65%  1.70251s         1  1.70251s  1.70251s  1.70251s  cudaEventSynchronize
                   42.12%  1.41588s         3  471.96ms  235.80ms  941.40ms  cudaMemcpy
                    5.71%  192.04ms         6  32.006ms  8.5770us  191.37ms  cudaMalloc
                    0.53%  17.884ms         1  17.884ms  17.884ms  17.884ms  cudaGetSymbolAddress
                   

In [21]:
!nvprof ./matmul_cublas 32768

Running cuBLAS matmul for N = 32768
==661== NVPROF is profiling process 661, command: ./matmul_cublas 32768
cuBLAS kernel time: 16808.104 ms
Works Correctly
==661== Profiling application: ./matmul_cublas 32768
==661== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   74.83%  16.7953s         1  16.7953s  16.7953s  16.7953s  volta_sgemm_128x64_nn
                   16.82%  3.77586s         1  3.77586s  3.77586s  3.77586s  [CUDA memcpy DtoH]
                    8.35%  1.87349s         2  936.74ms  932.62ms  940.86ms  [CUDA memcpy HtoD]
      API calls:   74.00%  16.7953s         1  16.7953s  16.7953s  16.7953s  cudaEventSynchronize
                   24.90%  5.65154s         3  1.88385s  932.85ms  3.77763s  cudaMemcpy
                    0.86%  194.15ms         6  32.358ms  4.7650us  193.30ms  cudaMalloc
                    0.07%  16.023ms         1  16.023ms  16.023ms  16.023ms  cudaGetSymbolAddress
                   

In [17]:
%%writefile matmulk.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>#include <iostream>
#include <iostream>

__global__ void matmul(float *fA,float *fB,float*fC,int N) {
    int row=threadIdx.x+blockDim.x*blockIdx.x;
    int col=threadIdx.y+blockDim.y*blockIdx.y;
    float sum=0.0f;
    if(row<N && col<N){
        for(int k=0;k<N;k++){
            sum+=fA[row*N+k]*fB[k*N+col];
        }
        fC[row*N+col]=sum;
    }
}

using namespace std;

int main() {
    int N=1024;
    dim3 threadPerBlock(32,32);
    int t=(N+31)/32;
    dim3 blocks(t,t);
    size_t size = N * N * sizeof(float);
    float *A=(float*)malloc(size);
    float *B=(float*)malloc(size);
    float *C=(float*)malloc(size);
    for(int i=0;i<N*N;i++){
        A[i]=1.0f;
        B[i]=1.0f;
    }
    
    cudaEvent_t estart,estop,fstart,fstop;
    cudaEventCreate(&estart);
    cudaEventCreate(&estop);
    cudaEventCreate(&fstart);
    cudaEventCreate(&fstop);
    cudaEventRecord(estart);

    float *fA,*fB,*fC;
    cudaMalloc(&fA,size);
    cudaMalloc(&fB,size);
    cudaMalloc(&fC,size);
    cudaMemcpy(fA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(fB,B,size,cudaMemcpyHostToDevice);

    cudaDeviceSynchronize();

    cudaEventRecord(fstart);
    matmul<<<blocks,threadPerBlock>>>(fA,fB,fC,N);
    cudaEventRecord(fstop);

    cudaMemcpy(C,fC,size,cudaMemcpyDeviceToHost);

    cudaEventRecord(estop);

    cudaDeviceSynchronize();

    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, fstart, fstop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, estart, estop);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);

    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(fabs(C[i*N+j]-N) > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(fA);
                cudaFree(fB);
                cudaFree(fC);
                free(A);
                free(B);
                free(C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(fA);
    cudaFree(fB);
    cudaFree(fC);
    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting matmulk.cu


In [15]:
!ls

matmul.c  matmul.cu  matmulk.cu


In [23]:
!nvcc matmulk.cu -o matmul

In [19]:
!nvprof ./matmul

==265== NVPROF is profiling process 265, command: ./matmul
Kernel Execution time: 190.630 ms
Total execution time on GPU: 197.716 ms
Works Correctly
==265== Profiling application: ./matmul
==265== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.74%  73.218ms         1  73.218ms  73.218ms  73.218ms  matmul(float*, float*, float*, int)
                    3.31%  2.5553ms         1  2.5553ms  2.5553ms  2.5553ms  [CUDA memcpy DtoH]
                    1.95%  1.5089ms         2  754.47us  752.70us  756.25us  [CUDA memcpy HtoD]
      API calls:   50.85%  212.11ms         4  53.028ms     557ns  212.11ms  cudaEventCreate
                   28.15%  117.41ms         1  117.41ms  117.41ms  117.41ms  cudaLaunchKernel
                   19.14%  79.817ms         3  26.606ms  971.28us  77.422ms  cudaMemcpy
                    1.24%  5.1771ms       228  22.706us     107ns  1.3683ms  cuDeviceGetAttribute
                    0.35%

In [22]:
%%writefile matmulk.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

__global__ void matmul(float *fA,float *fB,float*fC,int N) {
    int col=threadIdx.x+blockDim.x*blockIdx.x;
    int row=threadIdx.y+blockDim.y*blockIdx.y;
    float sum=0.0f;
    if(row<N && col<N){
        for(int k=0;k<N;k++){
            sum+=fA[row*N+k]*fB[k*N+col];
        }
        fC[row*N+col]=sum;
    }
}

using namespace std;

int main() {
    int N=1024;
    dim3 threadPerBlock(32,32);
    int t=(N+31)/32;
    dim3 blocks(t,t);
    size_t size = N * N * sizeof(float);
    float *A=(float*)malloc(size);
    float *B=(float*)malloc(size);
    float *C=(float*)malloc(size);
    for(int i=0;i<N*N;i++){
        A[i]=1.0f;
        B[i]=1.0f;
    }
    
    cudaEvent_t estart,estop,fstart,fstop;
    cudaEventCreate(&estart);
    cudaEventCreate(&estop);
    cudaEventCreate(&fstart);
    cudaEventCreate(&fstop);
    cudaEventRecord(estart);

    float *fA,*fB,*fC;
    cudaMalloc(&fA,size);
    cudaMalloc(&fB,size);
    cudaMalloc(&fC,size);
    cudaMemcpy(fA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(fB,B,size,cudaMemcpyHostToDevice);

    cudaDeviceSynchronize();

    cudaEventRecord(fstart);
    matmul<<<blocks,threadPerBlock>>>(fA,fB,fC,N);
    cudaEventRecord(fstop);

    cudaMemcpy(C,fC,size,cudaMemcpyDeviceToHost);

    cudaEventRecord(estop);

    cudaDeviceSynchronize();

    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, fstart, fstop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, estart, estop);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);

    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(fabs(C[i*N+j]-N) > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(fA);
                cudaFree(fB);
                cudaFree(fC);
                free(A);
                free(B);
                free(C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(fA);
    cudaFree(fB);
    cudaFree(fC);
    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting matmulk.cu


In [24]:
!nvprof ./matmul

==359== NVPROF is profiling process 359, command: ./matmul
Kernel Execution time: 29.344 ms
Total execution time on GPU: 35.327 ms
Works Correctly
==359== Profiling application: ./matmul
==359== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   66.30%  6.7562ms         1  6.7562ms  6.7562ms  6.7562ms  matmul(float*, float*, float*, int)
                   18.38%  1.8732ms         1  1.8732ms  1.8732ms  1.8732ms  [CUDA memcpy DtoH]
                   15.31%  1.5604ms         2  780.18us  770.14us  790.23us  [CUDA memcpy HtoD]
      API calls:   82.06%  187.09ms         4  46.773ms     529ns  187.09ms  cudaEventCreate
                    9.91%  22.586ms         1  22.586ms  22.586ms  22.586ms  cudaLaunchKernel
                    5.37%  12.234ms         3  4.0779ms  971.76us  10.269ms  cudaMemcpy
                    2.17%  4.9496ms       228  21.708us     110ns  1.4445ms  cuDeviceGetAttribute
                    0.26%  

In [ ]:
%%writefile matmulk.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

__global__ void matmul(float *fA,float *fB,float*fC,int N) {
    int col=threadIdx.x+blockDim.x*blockIdx.x;
    int row=threadIdx.y+blockDim.y*blockIdx.y;
    float sum=0.0f;
    if(row<N && col<N){
        for(int k=0;k<N;k++){
            sum+=fA[row*N+k]*fB[k*N+col];
        }
        fC[row*N+col]=sum;
    }
}

using namespace std;

int main() {
    int N=1024;
    dim3 threadPerBlock(32,32);
    int t=(N+31)/32;
    dim3 blocks(t,t);
    size_t size = N * N * sizeof(float);
    float *A=(float*)malloc(size);
    float *B=(float*)malloc(size);
    float *C=(float*)malloc(size);
    for(int i=0;i<N*N;i++){
        A[i]=1.0f;
        B[i]=1.0f;
    }
    
    cudaEvent_t estart,estop,fstart,fstop;
    cudaEventCreate(&estart);
    cudaEventCreate(&estop);
    cudaEventCreate(&fstart);
    cudaEventCreate(&fstop);
    cudaEventRecord(estart);

    float *fA,*fB,*fC;
    cudaMalloc(&fA,size);
    cudaMalloc(&fB,size);
    cudaMalloc(&fC,size);
    cudaMemcpy(fA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(fB,B,size,cudaMemcpyHostToDevice);

    cudaDeviceSynchronize();

    cudaEventRecord(fstart);
    matmul<<<blocks,threadPerBlock>>>(fA,fB,fC,N);
    cudaEventRecord(fstop);

    cudaMemcpy(C,fC,size,cudaMemcpyDeviceToHost);

    cudaEventRecord(estop);

    cudaDeviceSynchronize();

    float kernelmilliseconds = 0;
    cudaEventElapsedTime(&kernelmilliseconds, fstart, fstop);
    printf("Kernel Execution time: %.3f ms\n", kernelmilliseconds);

    float overallmilliseconds = 0;
    cudaEventElapsedTime(&overallmilliseconds, estart, estop);
    printf("Total execution time on GPU: %.3f ms\n", overallmilliseconds);

    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            if(fabs(C[i*N+j]-N) > 1e-5){
                fprintf(stderr, "Result verification failed at element %d!\n", i);
                cudaFree(fA);
                cudaFree(fB);
                cudaFree(fC);
                free(A);
                free(B);
                free(C);
                exit(EXIT_FAILURE);
            }
        }
    }
    cout<<"Works Correctly"<<endl;
    cudaFree(fA);
    cudaFree(fB);
    cudaFree(fC);
    free(A);
    free(B);
    free(C);

    return 0;
}
